# **BIO CLINICAL BERT TEXT ENCODER**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel


In [ ]:
BASE_PATH = "/content/drive/MyDrive/MIMIC_Extracted"

vitals_file = os.path.join(BASE_PATH, "biobart_output/vitals_biobart_text.csv")
notes_file  = os.path.join(BASE_PATH, "notes_output/notes_preprocessed.csv")

OUTPUT_PATH = os.path.join(BASE_PATH, "embeddings")
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
print("Loading text files...")

vitals_df = pd.read_csv(vitals_file)
notes_df = pd.read_csv(notes_file)

Loading text files...


In [ ]:
# ======================================================
# FIND COMMON ICU STAYS
# ======================================================

common_ids = sorted(
    set(vitals_df.icustay_id) &
    set(notes_df.icustay_id)
)

print("Common ICU stays:", len(common_ids))


vitals_df = vitals_df[vitals_df.icustay_id.isin(common_ids)].sort_values("icustay_id")
notes_df  = notes_df[notes_df.icustay_id.isin(common_ids)].sort_values("icustay_id")


Common ICU stays: 993


In [ ]:
print("\nLoading ClinicalBERT...")

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT"
)

model = AutoModel.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT"
).to(device)

model.eval()


Loading ClinicalBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
def encode_text(text):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    ).to(device)

    with torch.no_grad():
        output = model(**inputs)

    cls = output.last_hidden_state[:, 0, :]  # CLS token
    return cls.squeeze(0).cpu().numpy()

In [ ]:
# ENCODE BOTH MODALITIES

print("\nEncoding texts...")

vitals_embeddings = []
notes_embeddings = []

for i in tqdm(range(len(common_ids))):
    vitals_embeddings.append(
        encode_text(vitals_df.iloc[i]["vitals_text"])
    )

    notes_embeddings.append(
        encode_text(notes_df.iloc[i]["notes_text"])
    )


vitals_embeddings = np.stack(vitals_embeddings)
notes_embeddings = np.stack(notes_embeddings)


Encoding texts...


100%|██████████| 993/993 [54:58<00:00,  3.32s/it]


In [ ]:
np.savez_compressed(
    os.path.join(OUTPUT_PATH, "text_embeddings.npz"),
    ids=np.array(common_ids),
    vitals=vitals_embeddings,
    notes=notes_embeddings
)

print("\nEmbeddings saved!")
print("Shape vitals:", vitals_embeddings.shape)
print("Shape notes :", notes_embeddings.shape)


Embeddings saved!
Shape vitals: (993, 768)
Shape notes : (993, 768)


# **CROSS MODAL ATTENTION TRANSFORMER**

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from tqdm import tqdm

In [ ]:
DATA_PATH = "/content/drive/MyDrive/MIMIC_Extracted"  # change
OUTPUT_PATH = DATA_PATH

# ======================================================
# LOAD EMBEDDINGS
# ======================================================
data = np.load(os.path.join(DATA_PATH, "embeddings/text_embeddings.npz"))

ids = data["ids"]
vitals = torch.tensor(data["vitals"], dtype=torch.float32)
notes = torch.tensor(data["notes"], dtype=torch.float32)

print("Vitals:", vitals.shape)
print("Notes :", notes.shape)


Vitals: torch.Size([993, 768])
Notes : torch.Size([993, 768])


In [ ]:
class CrossModalTransformer(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, ff_dim=1024, dropout=0.1):
        super().__init__()

        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.t_proj = nn.Linear(embed_dim, embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, v, t):
        Q = self.v_proj(v).unsqueeze(1)
        K = self.t_proj(t).unsqueeze(1)
        V = self.t_proj(t).unsqueeze(1)

        attn_out, weights = self.attn(Q, K, V)

        x = self.norm1(Q + attn_out)
        ff_out = self.ff(x)
        fused = self.norm2(x + ff_out)

        return fused.squeeze(1), weights.squeeze(1)

In [ ]:
model = CrossModalTransformer()
model.eval()

print(model)

CrossModalTransformer(
  (v_proj): Linear(in_features=768, out_features=768, bias=True)
  (t_proj): Linear(in_features=768, out_features=768, bias=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (ff): Sequential(
    (0): Linear(in_features=768, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=768, bias=True)
  )
)


In [ ]:
print("\nGenerating fused representations...")

fused_list = []
weights_list = []

with torch.no_grad():
    for i in tqdm(range(len(ids))):
        f, w = model(vitals[i:i+1], notes[i:i+1])
        fused_list.append(f.cpu().numpy())
        weights_list.append(w.cpu().numpy())

fused_vectors = np.vstack(fused_list)
attention_weights = np.vstack(weights_list)

print("Fused shape:", fused_vectors.shape)


Generating fused representations...


100%|██████████| 993/993 [00:02<00:00, 353.33it/s]


Fused shape: (993, 768)


In [ ]:
np.savez_compressed(
    os.path.join(OUTPUT_PATH, "fused_embeddings.npz"),
    ids=ids,
    fused=fused_vectors,
    attention=attention_weights
)

print("\n Fused embeddings saved!")


 Fused embeddings saved!
